# Modelación en ingeniería
## Observador del estado: Filtro de Kalman

*Profesor: David Ortiz-Puerta*

---

> **💡 Nota para usuarios de Google Colab**
> 
> Antes de ejecutar el notebook, descarga las funciones auxiliares ejecutando la siguiente celda

In [ ]:
# !wget -q https://raw.githubusercontent.com/dortiz5/modelacion-en-ingenieria/main/Notebooks/helpers.py

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from helpers import (configure_plot_style, 
                        KF, ExKF, plot_kf_results, plot_ekf_results)

def plot_input(t, u, title=''):
    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.plot(t, u, lw=1.2)
    ax.set_xlabel('t [s]')
    ax.set_ylabel('u(t)')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

configure_plot_style()

def fmt_QR(Q, R):
    q_diag = np.diag(Q)
    return f"Q = diag({q_diag[0]:g}, {q_diag[1]:g}), R = {R[0, 0]:g}"

## Ejemplo lineal: satélite con acelerómetro

**Problema.** Un satélite mide su aceleración con un acelerómetro ruidoso. Se quiere estimar **posición y velocidad** a partir de una medición ruidosa de posición (radar).

**Modelo cinemático (continuo):**

$$\dot{p} = v, \qquad \dot{v} = u$$

**Discretización (Euler, paso $\Delta t$):**

$$\underbrace{\begin{pmatrix} p[n+1] \\ v[n+1] \end{pmatrix}}_{x[n+1]} = \underbrace{\begin{pmatrix} 1 & \Delta t \\ 0 & 1 \end{pmatrix}}_{A} \underbrace{\begin{pmatrix} p[n] \\ v[n] \end{pmatrix}}_{x[n]} + \underbrace{\begin{pmatrix} 0 \\ \Delta t \end{pmatrix}}_{B} u[n] + w[n]$$

**Medición (solo posición):**

$$y[n] = \underbrace{\begin{pmatrix} 1 & 0 \end{pmatrix}}_{C} \begin{pmatrix} p[n] \\ v[n] \end{pmatrix} + v[n]$$

**Observabilidad:** $\mathcal{O} = \begin{pmatrix} C \\ CA \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 1 & \Delta t \end{pmatrix}$, $\mathrm{rank}(\mathcal{O}) = 2$ $\Rightarrow$ **observable**.


In [ ]:
# Simulación: satélite con acelerómetro
dt = 0.1
T  = 20.0
N  = int(T / dt)

A = np.array([[1.0, dt],
              [0.0, 1.0]])
B = np.array([[0.0],
              [dt]])
C = np.array([[1.0, 0.0]])

Q_true = np.diag([0.01, 0.1])
R_true = np.array([[1.0]])

np.random.seed(0)
x_true  = np.zeros((N, 2))
y_meas  = np.zeros((N, 1))
u_input = np.ones((N, 1)) * 0.5

for n in range(N - 1):
    w = np.random.multivariate_normal([0, 0], Q_true)
    v = np.random.normal(0, np.sqrt(R_true[0, 0]))
    x_true[n+1] = A @ x_true[n] + (B @ u_input[n]).flatten() + w
    y_meas[n+1] = C @ x_true[n+1] + v

x0 = np.array([0.0, 0.0])
P0 = 10 * np.eye(2)


# Tres escenarios
# Caso 1: Q y R nominales
Q1, R1 = Q_true, R_true
x_h1, P_h1, K_h1 = KF(A, B, C, Q1, R1, u_input, y_meas, x0, P0)
fig1 = plot_kf_results(x_true, y_meas, x_h1, P_h1, K_h1, dt,
                       title=f"Caso 1 (nominal): {fmt_QR(Q1, R1)}")
plt.savefig('Satelite_1.pdf', format='pdf', bbox_inches='tight')
plt.show()

# Caso 2: R grande
R2 = np.array([[100.0]])
x_h2, P_h2, K_h2 = KF(A, B, C, Q1, R2, u_input, y_meas, x0, P0)
fig2 = plot_kf_results(x_true, y_meas, x_h2, P_h2, K_h2, dt,
                       title=f"Caso 2 (R grande, filtrado fuerte): {fmt_QR(Q1, R2)}")
plt.show()

# Caso 3: Q grande
Q3 = np.diag([1.0, 10.0])
x_h3, P_h3, K_h3 = KF(A, B, C, Q3, R1, u_input, y_meas, x0, P0)
fig3 = plot_kf_results(x_true, y_meas, x_h3, P_h3, K_h3, dt,
                       title=f"Caso 3 (Q grande, sigue mediciones): {fmt_QR(Q3, R1)}")
plt.show()

## Ejemplo no lineal: Lotka-Volterra

**Problema.** Se mide la población de presas (con ruido). Estimar también la de depredadores, no medida.

**Modelo (continuo):**

$$\dot{x}_1 = \alpha\,x_1 - \beta\,x_1\,x_2 \quad \text{(presas)}$$

$$\dot{x}_2 = \delta\,x_1\,x_2 - \gamma\,x_2 \quad \text{(depredadores)}$$

**Discretización (Euler, paso $\Delta t$):**

$$\begin{pmatrix} x_1[n+1] \\ x_2[n+1] \end{pmatrix} = \underbrace{\begin{pmatrix} x_1[n] + \Delta t\,(\alpha\,x_1[n] - \beta\,x_1[n]\,x_2[n]) \\ x_2[n] + \Delta t\,(\delta\,x_1[n]\,x_2[n] - \gamma\,x_2[n]) \end{pmatrix}}_{f(x[n])} + w[n]$$

**Medición:**

$$y[n] = \underbrace{\begin{pmatrix} 1 & 0 \end{pmatrix} x[n]}_{h(x[n])} + v[n]$$

**Jacobianos.** Linealizamos $f$ y $h$ en el estado estimado para obtener $A$ y $C$.

**Cálculo de** $A = \partial f / \partial x$:

$$A = I + \Delta t \begin{pmatrix} \dfrac{\partial f_1}{\partial x_1} & \dfrac{\partial f_1}{\partial x_2} \\ \dfrac{\partial f_2}{\partial x_1} & \dfrac{\partial f_2}{\partial x_2} \end{pmatrix} = I + \Delta t \begin{pmatrix} \alpha - \beta\,x_2 & -\beta\,x_1 \\ \delta\,x_2 & \delta\,x_1 - \gamma \end{pmatrix}$$

**Cálculo de** $C = \partial h / \partial x$:

$$C = \begin{pmatrix} \dfrac{\partial h}{\partial x_1} & \dfrac{\partial h}{\partial x_2} \end{pmatrix} = \begin{pmatrix} 1 & 0 \end{pmatrix}$$

**Observabilidad local** (en equilibrio $x^* = (\gamma/\delta,\,\alpha/\beta)$):

$$\mathcal{O} = \begin{pmatrix} C \\ C\,A(x^*) \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 1 & -\beta\,\Delta t\,x_1^* \end{pmatrix}, \quad \mathrm{rank}(\mathcal{O}) = 2 \Rightarrow \text{observable}$$

In [ ]:

alpha = 1.0
beta  = 0.5
delta = 0.5
gamma = 1.0
dt    = 0.01

def f_LV(x, u):
    x1, x2 = x[0], x[1]
    return np.array([
        x1 + dt * (alpha * x1 - beta * x1 * x2),
        x2 + dt * (delta * x1 * x2 - gamma * x2)
    ])


def df_dx_LV(x, u):
    x1, x2 = x[0], x[1]
    return np.eye(2) + dt * np.array([
        [alpha - beta * x2, -beta * x1],
        [delta * x2,         delta * x1 - gamma]
    ])


def h_LV(x, u):
    return np.array([x[0]])


def dh_dx_LV(x, u):
    return np.array([[1.0, 0.0]])

In [ ]:
# Simulación de "verdad" Lotka-Volterra
T = 20.0
N = int(T / dt)

Q_true = 1e-4 * np.eye(2)
R_true = np.array([[0.05]])

np.random.seed(0)
x_true  = np.zeros((N, 2))
y_meas  = np.zeros((N, 1))
u_input = np.zeros((N, 1))   # sin entrada externa

x_true[0] = np.array([1.5, 1.0])   # condición inicial

for n in range(N - 1):
    w = np.random.multivariate_normal([0, 0], Q_true)
    v = np.random.normal(0, np.sqrt(R_true[0, 0]))
    x_true[n+1] = f_LV(x_true[n], u_input[n]) + w
    y_meas[n+1] = h_LV(x_true[n+1], u_input[n+1]) + v

# Estado inicial del filtro (mal inicializado a propósito)
x0 = np.array([2.0, 1.0])
P0 = np.eye(2)


# Tres escenarios
# Caso 1: Q y R nominales
Q1, R1 = Q_true, R_true
x_h1, P_h1, K_h1 = ExKF(f_LV, h_LV, df_dx_LV, dh_dx_LV,
                       Q1, R1, u_input, y_meas, x0, P0)
fig1 = plot_ekf_results(x_true, y_meas, x_h1, P_h1, K_h1, dt,
                        title=f"Caso 1 (nominal): {fmt_QR(Q1, R1)}")
plt.savefig('Pred_pres.pdf', format='pdf', bbox_inches='tight')
plt.show()

# Caso 2: R grande
R2 = np.array([[10.0]])
x_h2, P_h2, K_h2 = ExKF(f_LV, h_LV, df_dx_LV, dh_dx_LV,
                       Q1, R2, u_input, y_meas, x0, P0)
fig2 = plot_ekf_results(x_true, y_meas, x_h2, P_h2, K_h2, dt,
                        title=f"Caso 2 (R grande, filtrado fuerte): {fmt_QR(Q1, R2)}")
plt.show()

# Caso 3: Q grande
Q3 = 1.0 * np.eye(2)
x_h3, P_h3, K_h3 = ExKF(f_LV, h_LV, df_dx_LV, dh_dx_LV,
                       Q3, R1, u_input, y_meas, x0, P0)
fig3 = plot_ekf_results(x_true, y_meas, x_h3, P_h3, K_h3, dt,
                        title=f"Caso 3 (Q grande, sigue mediciones): {fmt_QR(Q3, R1)}")
plt.show()